# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการป่าสุ่ม (Random Forests)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **ป่าสุ่ม (Random Forests)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองสำหรับจำแนกประเภทแบบสองกลุ่ม (Binary Classification) ที่มีความซับซ้อนและมีสัญญาณรบกวนสูง
2. ฝึกสอนแบบจำลอง **ต้นไม้ตัดสินใจเดี่ยว (Single Decision Tree)** เปรียบเทียบกับ **ป่าสุ่ม (Random Forest)** โดยใช้ไลบรารี `scikit-learn` เพื่อสังเกตพฤติกรรมว่าวิธีกลุ่มโมเดล (Ensembling) ช่วยปรับขอบเขตการตัดสินใจให้ราบเรียบขึ้นอย่างไร
3. สร้างแบบจำลอง **Random Forest จากศูนย์ (from scratch)** ด้วยกระบวนการสุ่มสุ่มตัวอย่างซ้ำแบบบูตสแตรป (Bootstrap Sampling) และการเลือกคุณลักษณะย่อยแบบสุ่ม (Random Feature Selection)
4. ประเมินผลความแม่นยำของโมเดลที่เขียนเองเปรียบเทียบกับไลบรารีมาตรฐาน scikit-learn
5. อธิบายแนวคิดกลุ่มโมเดล (Ensembling) ว่าเชื่อมโยงกับกระบวนการทำงานของ Deep Learning อย่างไร (เช่น Model Averaging หรือ Test Time Augmentation)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from collections import Counter

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลจำลอง (Data Generation)

เราจะสุ่มสร้างชุดข้อมูลรูปทรงพระจันทร์เสี้ยวสองวงประกบกันแบบมีสัญญาณรบกวน (Noisy Double Moons Dataset) ซึ่งมีลักษณะของขอบเขตที่ไม่เป็นเชิงเส้นและมีความเหลื่อมล้ำซ้อนทับกัน เพื่อจำลองพื้นที่ของคุณลักษณะที่มีความคลุมเครือ ซึ่งพบได้บ่อยในปัญหาของระบบภาพคอมพิวเตอร์ (Computer Vision) ในโลกจริง

In [ ]:
# สร้างชุดข้อมูลจำลองแบบ Double Moons ที่มีสัญญาณรบกวนสูง
X_train, y_train = make_moons(n_samples=150, noise=0.35, random_state=42)
X_test, y_test = make_moons(n_samples=100, noise=0.35, random_state=42)

# พล็อตกราฟแสดงจุดกระจายตัวของชุดข้อมูลจำลอง
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='red', label='Class 0: Handwheel Valve', alpha=0.7)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='blue', label='Class 1: Lever Valve', alpha=0.7)
plt.xlabel('Visual Feature 1')
plt.ylabel('Visual Feature 2')
plt.title('PTT Component Classification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. เปรียบเทียบพื้นที่ขอบเขตของต้นไม้ตัดสินใจเดี่ยวกับป่าสุ่ม

เราลองมาวาดภาพเปรียบเทียบขอบเขตพื้นที่การทำนายผลลัพธ์ระหว่างแบบจำลองต้นไม้ตัดสินใจต้นเดี่ยว (Single Decision Tree) กับแบบจำลองป่าสุ่มที่มีการใช้จำนวนต้นไม้ต่างกัน เพื่อดูประสิทธิภาพในการแบ่งประเภท

In [ ]:
# สร้างจุดโครงข่ายข้อมูล Grid เพื่อนำไปวาดขอบเขตแบ่งประเภท
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# แบบจำลองต่างๆ ที่จะนำมาเปรียบเทียบ
models = {
    'Single Decision Tree': DecisionTreeClassifier(max_depth=None, random_state=42),
    'Random Forest (10 Trees)': RandomForestClassifier(n_estimators=10, random_state=42),
    'Random Forest (100 Trees)': RandomForestClassifier(n_estimators=100, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['red', 'blue']

for idx, (name, clf) in enumerate(models.items()):
    clf.fit(X_train, y_train)
    Z = clf.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    # ประเมินค่าความถูกต้องบนชุดข้อมูลทดสอบ
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    
    ax = axes[idx]
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=35)
    ax.set_title(f'{name}\nTest Accuracy: {test_acc * 100:.1f}%')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

*   **ต้นไม้ตัดสินใจต้นเดี่ยว (Single Decision Tree):** เส้นขอบเขตมีความซับซ้อนและหยักแหลมสูงมาก รวมถึงเกิดดินแดนเฉพาะตัวที่เป็นจุดเกาะเล็กเกาะน้อยลอยแยกต่างหาก (Isolated Islands) ซึ่งเป็นอาการจำลองข้อมูลเกินจริง (Overfitting) เข้ากับจุดสัญญาณรบกวน
*   **ป่าสุ่ม 100 ต้น (Random Forest - 100 Trees):** เส้นแบ่งระหว่างฝั่งมีความราบเรียบและยืดหยุ่นสูง โค้งรับไปตามโครงสร้างโค้งรูปพระจันทร์สองวงดั้งเดิมได้อย่างสอดคล้อง ส่งผลให้ค่าความแม่นยำของข้อมูลทดสอบ (Test Accuracy) ดีขึ้นอย่างเด่นชัด

## 3. การสร้าง Random Forest จากศูนย์ด้วย NumPy (Random Forest from Scratch)

เรามาสร้างแบบจำลองป่าสุ่มขึ้นมาด้วยตนเอง โดยในแต่ละต้นไม้ในป่าจะใช้ขั้นตอนหลักดังนี้:
1.  **Bootstrap Sampling:** ทำการสุ่มดึงตัวอย่างข้อมูลจากจุดข้อมูลทั้งหมดจำนวน $m$ แถวโดยอนุญาตให้มีตัวอย่างซ้ำกันได้
2.  **Random Feature Selection:** ทำการสุ่มเลือกคอลัมน์ของคุณลักษณะย่อยแบบสุ่มในแต่ละต้นไม้เพื่อเพิ่มความหลากหลายและลดความเชื่อมโยงระหว่างต้นไม้ในป่า
3.  ฝึกสอนแบบจำลองต้นไม้ตัดสินใจปกติลงบนชุดข้อมูลย่อยนั้น
4.  รวมผลลัพธ์การทำนายจากทุกต้นไม้เพื่อลงมติข้อสรุปสุดท้ายด้วยการลงคะแนนเสียงข้างมาก (Majority Voting)

In [ ]:
class CustomRandomForest:
    def __init__(self, n_estimators=10, max_depth=3, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []
        self.feat_indices = []

    def _bootstrap_samples(self, X, y):
        m = X.shape[0]
        indices = np.random.choice(m, m, replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        self.feat_indices = []
        m, n = X.shape
        
        # กำหนดจำนวนของคุณลักษณะที่จะสุ่มเลือก
        if self.max_features is None:
            n_features_to_sample = n
        elif self.max_features == 'sqrt':
            n_features_to_sample = int(np.sqrt(n))
        else:
            n_features_to_sample = self.max_features
            
        for _ in range(self.n_estimators):
            # 1. ทำการสุ่มบูตสแตรปข้อมูล (Bootstrap)
            X_boot, y_boot = self._bootstrap_samples(X, y)
            
            # 2. สุ่มเลือกดัชนีคอลัมน์คุณลักษณะย่อย
            feat_idx = np.random.choice(n, n_features_to_sample, replace=False)
            self.feat_indices.append(feat_idx)
            
            # 3. เทรนต้นไม้ตัดสินใจบนชุดข้อมูลย่อยตามคุณลักษณะที่สุ่มมา
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X_boot[:, feat_idx], y_boot)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = []
        for tree, feat_idx in zip(self.trees, self.feat_indices):
            preds = tree.predict(X[:, feat_idx])
            tree_preds.append(preds)
            
        tree_preds = np.array(tree_preds).T
        
        # ลงคะแนนเสียงข้างมากเพื่อเป็นคำทำนายสรุป
        final_preds = np.array([Counter(row).most_common(1)[0][0] for row in tree_preds])
        return final_preds

# เทรนป่าสุ่มที่เราสร้างขึ้นมาเองจากศูนย์
forest_scratch = CustomRandomForest(n_estimators=100, max_depth=5, max_features=2)
forest_scratch.fit(X_train, y_train)

# ประเมินผลแบบจำลองป่าสุ่มเชิงคณิตศาสตร์ของเรา
y_pred_scratch = forest_scratch.predict(X_test)
scratch_acc = accuracy_score(y_test, y_pred_scratch)

print(f"Scratch Random Forest Test Accuracy: {scratch_acc * 100:.2f}%")

## 4. พล็อตแสดงขอบเขตการตัดสินใจของป่าสุ่มแบบกำหนดเอง

มาลองวาดขอบเขตการตัดสินใจของแบบจำลอง Custom Random Forest ที่เราเทรนและสร้างขึ้นมาเองจากสัญชาตญาณกันครับ

In [ ]:
Z_scratch = forest_scratch.predict(grid_points)
Z_scratch = Z_scratch.reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z_scratch, cmap=cmap_light, alpha=0.6)
plt.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=40)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'Scratch Random Forest Boundary (100 Trees)\nTest Accuracy: {scratch_acc * 100:.1f}%')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การเฉลี่ยผลรวมกลุ่มโครงข่าย (Ensemble Averaging):** ในโปรเจกต์ระดับการผลิตจริงหรือการแข่งขันชิงรางวัล การสร้างโมเดลโครงข่ายประสาทชุดเดียวกันขึ้นมาหลาย ๆ ตัว (เช่น เทรนโมเดล YOLO 5 ชุดด้วยค่าเริ่มต้น Seed ที่ต่างกัน) แล้วนำผลลัพธ์พิกัด Bounding Boxes ที่ทำนายได้มาหาค่าเฉลี่ยร่วมกัน เป็นวิธีมาตรฐานที่ช่วยเพิ่มความปลอดภัยในการใช้งานสูงสุดและแก้ปัญหาน้ำหนักเอนเอียงได้ดี
*   **Test Time Augmentation (TTA):** ในกระบวนการทำนายผลลัพธ์จริง เราสามารถส่งภาพนำเข้าไปดัดแปลงแบบสุ่มหลายๆ รูปแบบ (เช่น กลับรูปซ้ายขวา ขยายขนาดภาพ หรือปรับเฉดสี) เข้าไปทำนายผลบน YOLO ตัวเดิม แล้วรวมมติพิกัดและค่าความมั่นใจกลับมารวมกัน วิธีนี้เปรียบเสมือนการสร้างกลุ่มโมเดลจำลองเสมือน (Virtual Ensemble) ที่ช่วยให้การทำนายพิกัดกล่องมีความนิ่งและมั่นคง เช่นเดียวกับแนวคิดของการสุ่มเก็บถุงข้อมูลซ้ำ (Bagging) ใน Random Forest